In [ ]:
#@title Run this cell first -- homework checking setup { display-mode: "form" }
#@markdown **Run this cell before anything else**, every time you open the notebook.
#@markdown <br><br>It sets up `grader.check(...)` so you can check your answers as you go.
#@markdown Nothing is installed and nothing is downloaded -- this notebook is
#@markdown self-contained, so it works even if Colab or GitHub is having a bad day.

import base64
import datetime
import doctest
import json
import sys

RECEIPT_PREFIX = "CS1910-RECEIPT:"

_HTML = """<div style="font-family:system-ui,sans-serif;font-size:14px;margin:4px 0;
 padding:10px 12px;border-left:5px solid {bar};background:{bg};border-radius:4px">
 <b>{name}</b> &mdash; {earned}/{possible} {verdict}{detail}</div>"""


class Result:
    """One question's outcome. Displays as HTML in Colab, as text everywhere else.

    The text form is what lands in the notebook's saved output and what the grader
    parses, so keep the first line machine-readable.
    """

    def __init__(self, name, earned, possible, messages):
        self.name, self.earned, self.possible = name, earned, possible
        self.messages = messages

    @property
    def passed(self):
        return self.earned == self.possible

    def __repr__(self):
        head = "{}: {}/{} {}".format(self.name, self.earned, self.possible,
                                     "PASSED" if self.passed else "FAILED")
        return "\n".join([head] + self.messages)

    def _repr_html_(self):
        bar, bg = ("#2e7d32", "#edf7ed") if self.passed else ("#c62828", "#fdecea")
        verdict = "passed" if self.passed else "not passing yet"
        detail = ""
        if self.messages:
            body = "\n".join(self.messages).replace("&", "&amp;").replace("<", "&lt;")
            detail = ('<pre style="margin:8px 0 0;white-space:pre-wrap;font-size:12px;'
                      'color:#444">{}</pre>'.format(body))
        return _HTML.format(bar=bar, bg=bg, name=self.name, earned=self.earned,
                            possible=self.possible, verdict=verdict, detail=detail)


def run_case(code, globs):
    """Run one doctest string. Returns (passed, report_text).

    `get_doctest` copies `globs`, so a case that rebinds a name cannot leak that
    change back into the student's namespace or into the next case.
    """
    parser = doctest.DocTestParser()
    test = parser.get_doctest(code.rstrip() + "\n", dict(globs), "case", None, 0)
    runner = doctest.DocTestRunner(verbose=False,
                                   optionflags=doctest.NORMALIZE_WHITESPACE)
    out = []
    try:
        runner.run(test, out=out.append, clear_globs=False)
    except Exception as exc:                      # a case that raises is a failure
        return False, "{}: {}".format(type(exc).__name__, exc)
    return runner.failures == 0, "".join(out)


def score(name, spec, globs):
    """Score one question against `globs`. Returns a Result."""
    cases = spec["suites"][0]["cases"] if "suites" in spec else spec["cases"]
    per_case = bool(cases) and all("points" in c for c in cases)

    earned = possible = 0
    failures, notes = [], []
    for case in cases:
        ok, report = run_case(case["code"], globs)
        if per_case:
            possible += case["points"]
            earned += case["points"] if ok else 0
        if ok:
            if case.get("success_message"):
                notes.append(case["success_message"])
        else:
            failures.append(case.get("failure_message") or _trim(report))

    if not per_case:
        possible = spec.get("points", len(cases))
        earned = 0 if failures else possible

    return Result(name, earned, possible, failures or notes)


def _trim(report, limit=600):
    """Doctest reports are verbose; keep the part a first-year student can act on."""
    text = report.strip()
    return text if len(text) <= limit else text[:limit] + "\n... (truncated)"


def caller_namespace():
    """Where the student's variables live.

    In Colab and Jupyter that is the interactive namespace, which is right even when
    `check()` is called from inside a function. Outside IPython -- the headless grader,
    or a plain `python` run -- fall back to the calling frame's globals, which is the
    namespace `exec` was handed.
    """
    try:
        shell = get_ipython()           # noqa: F821  (injected by IPython)
        if shell is not None:
            return shell.user_ns
    except NameError:
        pass
    return sys._getframe(2).f_globals


class Grader:
    """Holds the tests for one homework and remembers what has been checked.

    Named `grader` in the student notebook so existing `grader.check("q1")` cells
    keep working untouched.
    """

    def __init__(self, hw, tests):
        self.hw, self.tests, self.log = hw, tests, {}

    def check(self, name, globs=None):
        if name not in self.tests:
            raise KeyError("no test named {!r} in {} (have: {})".format(
                name, self.hw, ", ".join(sorted(self.tests))))
        result = score(name, self.tests[name], globs or caller_namespace())
        self.log[name] = (result.earned, result.possible)
        return result

    def check_all(self, globs):
        """Score every question against one namespace. This is what grading uses."""
        results = [score(n, self.tests[n], globs) for n in sorted(self.tests)]
        self.log = {r.name: (r.earned, r.possible) for r in results}
        return results

    def summary(self):
        return {"hw": self.hw,
                "results": {k: list(v) for k, v in sorted(self.log.items())},
                "total": sum(e for e, _ in self.log.values()),
                "possible": sum(p for _, p in self.log.values())}

    def receipt(self, student_id="", extra=None):
        """Print the one-line machine-readable receipt and return it.

        The grader re-runs the tests itself, so this line is a convenience for the
        student and a fast triage signal -- never the authoritative score.
        """
        body = self.summary()
        body["id"] = str(student_id)
        body["t"] = datetime.datetime.now().isoformat(timespec="seconds")
        if extra:
            body.update(extra)
        blob = base64.b64encode(
            json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
        ).decode()
        line = RECEIPT_PREFIX + blob
        print(line)
        return line


def parse_receipt(text):
    """Pull the last receipt out of arbitrary text. Returns a dict, or None."""
    found = None
    for chunk in text.split(RECEIPT_PREFIX)[1:]:
        token = chunk.split()[0] if chunk.split() else ""
        try:
            found = json.loads(base64.b64decode(token))
        except Exception:
            continue
    return found


def load_ok_tests(directory):
    """Read a directory of `q*.py` test files (OK_FORMAT doctests) into {name: spec}."""
    import pathlib
    tests = {}
    for path in sorted(pathlib.Path(directory).glob("*.py")):
        namespace = {}
        exec(compile(path.read_text(encoding="utf-8"), str(path), "exec"), namespace)
        spec = namespace["test"]
        tests[spec["name"]] = spec
    return tests

_TESTS = {'q1': {'name': 'q1',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> clean_handle('  @MiaCodes ')\n'miacodes'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> clean_handle('Mia_Codes')\n'mia_codes'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> clean_handle('@@ghost')\n'@ghost'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> clean_handle('@')\n''",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q2': {'name': 'q2',
        'points': 3,
        'suites': [{'cases': [{'code': ">>> q2_first == 'c' and q2_last == 'i'\nTrue",
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': ">>> q2_school == 'upei'\nTrue",
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': ">>> q2_backwards == 'iepu_0191sc'\nTrue",
                               'hidden': False,
                               'locked': False,
                               'points': 1}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q3': {'name': 'q3',
        'points': 1,
        'suites': [{'cases': [{'code': ">>> q3_option.strip().upper() == 'B'\nTrue",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q4': {'name': 'q4',
        'points': 1,
        'suites': [{'cases': [{'code': '>>> q4_case_gap == 32\nTrue',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q5': {'name': 'q5',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> hashtag_count('love this #fyp #upei')\n2",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> hashtag_count('no tags here')\n0",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> hashtag_count('#a#b#c')\n3",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> hashtag_count('')\n0",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q6': {'name': 'q6',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> first_tag('hey #fyp more')\n'fyp'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> first_tag('hey #fyp')\n'fyp'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> first_tag('#upei rocks')\n'upei'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> first_tag('nothing tagged here')\n''",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> first_tag('hey # there')\n''",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]}}

grader = Grader('09-Sequences-Handles', _TESTS)
check = grader.check

print("Ready. 09-Sequences-Handles loaded: 6 question(s), 11 point(s).")
print("Check a question any time with, for example:  grader.check(\"q1\")")

## Before you begin

### Submission
Before you start working on this Homework:

1. Press **File->Save a copy in Drive** from the menu. This will save your own copy of this notebook

2. Follow the steps below and ensure you run the cells in order from the top. (You'll have to ensure you run any cells above this one too)

3. You should get into the habit of saving your file occassionally as you work

4. Certain cells will check your work against the expected answer, when you run these cells you'll know if your work is correct.

5. After all the changes are done and progress is saved **File->Download->Download.ipynb** and then upload the .ipynb file to moodle

# Homework: Sequences — Handle Check

You are building the bit of a social app that deals with **text**: the handle someone
types when they sign up, and the caption they write on a post.

Text is messy. People paste their handle with a stray space on the end, or with the `@`
still attached, or in a mix of upper and lower case — and all three are the same person.
Captions are worse: hashtags can be anywhere in them, jammed together, or absent.

A string is a **sequence**: an ordered run of characters you can index into, slice, and
walk through one at a time. Everything in this homework is one of those three moves.

## Question 1 — Tidy up a handle

Two people should not be able to claim `@MiaCodes` and `@miacodes` as different handles.
Before we store a handle we **normalise** it, which means turning every way of writing it
into one canonical form.

Write `clean_handle(raw)` that does three things, in this order:

1. removes whitespace from both ends
2. removes **one** leading `@`, if there is one
3. converts the result to lower case

```
clean_handle('  @MiaCodes ')  ->  'miacodes'
clean_handle('Mia_Codes')     ->  'mia_codes'      no @ to remove
clean_handle('@@ghost')       ->  '@ghost'         only the first @ goes
clean_handle('@')             ->  ''               nothing left
```

The string methods from lecture — `.strip()`, `.lower()`, `.startswith()` — plus a slice
are all you need. Remember that string methods **return a new string**; they never change
the one you called them on.

In [ ]:
def clean_handle(raw):
    ...

# try it out
print(clean_handle('  @MiaCodes '))

In [ ]:
grader.check("q1")

## Question 2 — Index it, slice it, flip it

The course account's handle is:

```python
handle = 'cs1910_upei'
```

Fill in four variables below **using indexing and slicing on `handle`** — do not type the
answers as literal strings, and do not count characters by hand in your final answer.

| Variable | What it should hold |
|---|---|
| `q2_first` | the first character |
| `q2_last` | the last character, found with a **negative** index |
| `q2_school` | everything from the `u` to the end — the string `'upei'` |
| `q2_backwards` | the whole handle reversed |

For `q2_backwards`, remember that a slice takes a **step** as its third value, and that a
step of `-1` walks the sequence backwards.

Each row is worth 1 point, so a partly-right answer still earns something.

In [ ]:
handle = 'cs1910_upei'

q2_first = ...
q2_last = ...
q2_school = ...
q2_backwards = ...

print(q2_first, q2_last, q2_school, q2_backwards)

In [ ]:
grader.check("q2")

## Question 3 — Why can't I just fix the capital letter?

A classmate wants to capitalise the first letter of a handle, so they write:

```python
handle = 'miacodes'
handle[0] = 'M'
```

What actually happens when that second line runs?

**A** — `handle` becomes `'Miacodes'`

**B** — Python raises a `TypeError`, because strings cannot be changed in place

**C** — nothing happens; Python ignores the line silently

Set `q3_option` to `'A'`, `'B'` or `'C'`.

(If you are not sure, try it in a scratch cell and read the error. Reading errors is a
skill, not a punishment.)

In [ ]:
q3_option = ...

In [ ]:
grader.check("q3")

## Question 4 — Why does `'Zoe'` come before `'aaron'`?

Sort a list of handles and you will find every capitalised one lands ahead of every
lower-case one. That is not a bug. Python compares strings character by character using
each character's **ordinal** — its number in the Unicode table — and every capital letter
has a smaller ordinal than every lower-case letter.

The `ord()` function gives you that number. Run `ord('a')` and `ord('A')` and look at the
gap between them.

Set `q4_case_gap` to the result of `ord('a') - ord('A')` — as a number you have worked
out, not as the expression itself.

In [ ]:
# a scratch cell to explore with
print(ord('A'), ord('a'), ord('Z'), ord('z'))

q4_case_gap = ...

In [ ]:
grader.check("q4")

## Question 5 — Count the hashtags

Write `hashtag_count(caption)`, which returns how many `'#'` characters are in a caption.

```
hashtag_count('love this #fyp #upei')  ->  2
hashtag_count('no tags here')          ->  0
hashtag_count('#a#b#c')                ->  3
hashtag_count('')                      ->  0
```

Walk the caption with a `for` loop, one character at a time, and keep a counter.

(Python has a `.count()` method that would do this in one line. Write the loop anyway —
the loop is what the next question needs you to be comfortable with.)

In [ ]:
def hashtag_count(caption):
    ...

# try it out
print(hashtag_count('love this #fyp #upei'))

In [ ]:
grader.check("q5")

## Question 6 — Pull out the first hashtag

Write `first_tag(caption)`, which returns the **first hashtag in the caption, without the
`#`**. If the caption has no hashtag, return the empty string `''`.

A hashtag runs from the `#` up to the next space, or to the end of the caption if there
is no space after it.

```
first_tag('hey #fyp more')        ->  'fyp'
first_tag('hey #fyp')             ->  'fyp'      runs to the end of the string
first_tag('#upei rocks')          ->  'upei'
first_tag('nothing tagged here')  ->  ''         no # at all
first_tag('hey # there')          ->  ''         a # with nothing attached to it
```

This is a decomposition problem — take it in pieces:

1. Where is the `#`? `.find()` returns its index, or `-1` if it is not there at all.
2. Slice off everything after it.
3. Where does that slice's first space fall? `.find()` again, and `-1` again means
   "no space, so the tag runs to the end".
4. Slice again to cut the tag out.

That last case, `'hey # there'`, falls out of the rules on its own — if your code follows
steps 1–4 you do not need a special case for it.

In [ ]:
def first_tag(caption):
    ...

# try it out
print(first_tag('hey #fyp more'))

In [ ]:
grader.check("q6")

## Beyond the homework

Not graded:

- Extend `first_tag` into `all_tags(caption)`, returning every hashtag. You will want a
  loop around what you already wrote.
- A handle is only valid if it is 3–15 characters and uses letters, digits and
  underscores only. Write `valid_handle(handle)`.
- `clean_handle('@@ghost')` gives `'@ghost'`, which is not a legal handle at all. Should
  normalising and validating be the same function, or two? Argue both sides.

# Submission

Save a copy in Drive to ensure you have your own copy of this notebook and to ensure that you've saved your latest work.

- Ensure you run all the cells in order from the top to the bottom
- Run the final cell to see your total and produce your receipt
- **File->Download->Download .ipynb** and upload the `.ipynb` file to Moodle

Your mark comes from your instructor re-running this notebook, not from the receipt
line -- so make sure the notebook itself is saved with your work in it.

In [ ]:
#@title Finished? Run this to see your total { display-mode: "form" }
#@markdown Re-checks every question and prints your score, plus a receipt line.
#@markdown <br><br>Your grade comes from your instructor re-running this notebook,
#@markdown so the receipt is for your records -- make sure the notebook itself is saved.

student_id = ""  #@param {type:"string"}

for _r in grader.check_all(globals()):
    print(_r)
_s = grader.summary()
print("\nTOTAL: {}/{}".format(_s["total"], _s["possible"]))
grader.receipt(student_id)
